In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

In [ ]:
# 1) Create 100 one-liner texts
texts = [
    "AI models are getting larger every year",
    "Coffee prices fluctuate with global demand",
    "Python loops make repetitive tasks easier",
    "SQL joins merge data across related tables",
    "The best time to email customers is afternoon",
    "Vector databases improve semantic search",
    "Transfer learning reduces training time",
    "Open source drives faster innovation",
    "Cosine similarity measures text closeness",
    "Apple launches new devices every September",
    "Pandas groupby aggregates large datasets",
    "Feature engineering improves model accuracy",
    "Normalization helps models converge faster",
    "Random forests combine many decision trees",
    "Gradient boosting refines weak learners",
    "K-means clustering groups similar points",
    "Dimensionality reduction speeds up training",
    "Visualization helps explain complex data",
    "Streaming data needs real-time processing",
    "Outlier detection prevents model bias",
    "Cross-validation improves generalization",
    "Hyperparameter tuning optimizes accuracy",
    "Batch processing handles large datasets",
    "Cloud storage scales automatically",
    "Data pipelines automate ETL workflows",
    "Machine learning powers recommendation systems",
    "Text embeddings convert words into vectors",
    "Image embeddings capture visual similarity",
    "Speech recognition converts sound to text",
    "Transformers replaced RNNs in NLP",
    "Attention mechanism focuses on context",
    "Reinforcement learning learns by reward",
    "Synthetic data helps balance datasets",
    "Time series forecasting predicts demand",
    "Business intelligence helps decision making",
    "Interactive dashboards improve insights",
    "A/B testing measures campaign success",
    "Customer segmentation improves marketing",
    "Fraud detection protects financial systems",
    "Personalization improves customer experience",
    "Deep learning requires large datasets",
    "Model evaluation ensures reliability",
    "Explainability builds model trust",
    "Ethics in AI ensures responsible innovation",
    "Data governance ensures data quality",
    "API integration connects multiple systems",
    "Microservices improve scalability",
    "Docker containers simplify deployment",
    "CI/CD pipelines automate software releases",
    "Serverless computing optimizes cost",
    "Edge AI brings inference closer to devices",
    "Federated learning protects privacy",
    "Prompt engineering improves LLM responses",
    "Tokenization breaks text into subwords",
    "Embeddings capture semantic meaning",
    "Vector search retrieves similar content",
    "RAG combines retrieval with generation",
    "LangChain simplifies LLM orchestration",
    "Agents coordinate multiple AI tasks",
    "Pinecone indexes large vector datasets",
    "Chroma provides lightweight vector storage",
    "Faiss accelerates approximate nearest neighbor search",
    "Sentence transformers create text embeddings",
    "Fine-tuning customizes pretrained models",
    "Quantization reduces model size",
    "Distillation transfers knowledge to smaller models",
    "LoRA fine-tunes large models efficiently",
    "RLHF aligns models with human feedback",
    "GPT models use autoregressive generation",
    "BERT uses bidirectional attention",
    "LLMs understand context and nuance",
    "OpenAI advances general-purpose AI",
    "Google trains foundation models at scale",
    "Meta focuses on open research models",
    "Anthropic emphasizes safety and alignment",
    "NVIDIA optimizes GPUs for deep learning",
    "Transformer architecture scales efficiently",
    "Self-attention connects distant tokens",
    "Batch normalization stabilizes learning",
    "Dropout prevents overfitting",
    "ReLU activation speeds convergence",
    "Adam optimizer adapts learning rates",
    "Learning rate scheduling improves training",
    "Data augmentation improves robustness",
    "Model checkpoints save progress",
    "Early stopping prevents overtraining",
    "Evaluation metrics guide improvement",
    "Precision and recall measure performance",
    "ROC curve visualizes trade-offs",
    "Confusion matrix summarizes predictions",
    "F1 score balances precision and recall",
    "AUC measures classification quality",
    "Regression minimizes squared errors",
    "Classification predicts categorical outcomes",
    "Clustering groups unlabeled data",
    "Anomaly detection identifies unusual patterns",
    "Dimensionality reduction simplifies data visualization",
    "Principal component analysis reduces redundancy",
    "t-SNE projects data into 2D space",
    "UMAP preserves local relationships",
    "Word2Vec learns from co-occurrence statistics",
    "Doc2Vec represents full sentences as vectors",
    "TF-IDF weighs word importance in documents",
    "BM25 scores document relevance in search engines",
]

In [ ]:
# STEP 1 — initialize vectordb (in-memory) + config

# simple doc/vec stores
doc_db = {}   # id -> raw text
vec_db = {}   # id -> vector (optional mirror)

# embedding model
model_name = "sentence-transformers/paraphrase-MiniLM-L6-v2"
model = SentenceTransformer(model_name)
dim = model.get_sentence_embedding_dimension()  # e.g., 384 for MiniLM

# HNSW params
M   = 16     # max neighbors per node
efC = 100    # build effort
efS = 64     # search effort
metric = faiss.METRIC_INNER_PRODUCT  # use cosine via L2-normalization + IP

In [ ]:
# STEP 2 — create index (if it doesn't exist)

# Base HNSW index; some FAISS builds don't accept metric in ctor → fallback
base = faiss.IndexHNSWFlat(dim, M, metric)

base.hnsw.efConstruction = efC
base.hnsw.efSearch = efS

# Wrap with ID map to use your own ids
index = faiss.IndexIDMap2(base)

In [ ]:
base

In [ ]:
#Step 3
def get_index():
    # in real life you might load from disk; here we return the in-mem object
    return index

faiss_index = get_index()
faiss_index

In [ ]:
# STEP 4 — organize documents as [{id, doc}]

doc_records = [{"id": i, "doc": txt} for i, txt in enumerate(texts)]
for r in doc_records:
    doc_db[r["id"]] = r["doc"]

len(doc_records), doc_records[0]

In [ ]:
# STEP 5 — create embeddings and organize as [{id, [embedding]}]

X = model.encode([r["doc"] for r in doc_records], convert_to_numpy=True).astype(np.float32)

# normalize so IP == cosine (and L2 distance ranking is equivalent on unit vectors)
faiss.normalize_L2(X)

ids = np.array([r["id"] for r in doc_records], dtype=np.int64)

# optional mirror
for pid, vec in zip(ids, X):
    vec_db[int(pid)] = vec

len(ids), X.shape

In [ ]:
faiss_index.add_with_ids(X, ids)

In [ ]:
faiss_index.ntotal

In [ ]:
faiss_index

In [ ]:
X[0]

In [ ]:
ps = faiss.ParameterSpace()

def search(query, topk=5, ef_search=None):
    if ef_search is not None:
        ps.set_index_parameter(index, "efSearch", int(ef_search))  # works through wrappers

    q = model.encode([query], convert_to_numpy=True).astype(np.float32)
    faiss.normalize_L2(q)
    D, I = index.search(q, topk)
    out = []
    for score, pid in zip(D[0], I[0]):
        if pid != -1:
            out.append((float(score), int(pid), doc_db[int(pid)]))
    return out


# try a couple of queries
for q in ["semantic vector search over documents",
          "Apple product announcements",
          "best time to contact customers"]:
    print("\nQuery:", q)
    for s, pid, txt in search(q, topk=5, ef_search=64):
        print(f"  score={s:.3f}  id={pid}  -> {txt}")

In [1]:
### IVF 
## IVF
# STEP 0: imports (run once)
# pip install -q faiss-cpu sentence-transformers
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer


# ---------- 1) Data & embeddings ----------
texts = [
    "AI models are getting larger every year",
    "Coffee prices fluctuate with global demand",
    "Python loops make repetitive tasks easier",
    "SQL joins merge data across related tables",
    "The best time to email customers is afternoon",
    "Vector databases improve semantic search",
    "Transfer learning reduces training time",
    "Open source drives faster innovation",
    "Cosine similarity measures text closeness",
    "Apple launches new devices every September",
    "Pandas groupby aggregates large datasets",
    "Feature engineering improves model accuracy",
    "Normalization helps models converge faster",
    "Random forests combine many decision trees",
    "Gradient boosting refines weak learners",
    "K-means clustering groups similar points",
    "Dimensionality reduction speeds up training",
    "Visualization helps explain complex data",
    "Streaming data needs real-time processing",
    "Outlier detection prevents model bias",
    "Cross-validation improves generalization",
    "Hyperparameter tuning optimizes accuracy",
    "Batch processing handles large datasets",
    "Cloud storage scales automatically",
    "Data pipelines automate ETL workflows",
    "Machine learning powers recommendation systems",
    "Text embeddings convert words into vectors",
    "Image embeddings capture visual similarity",
    "Speech recognition converts sound to text",
    "Transformers replaced RNNs in NLP",
    "Attention mechanism focuses on context",
    "Reinforcement learning learns by reward",
    "Synthetic data helps balance datasets",
    "Time series forecasting predicts demand",
    "Business intelligence helps decision making",
    "Interactive dashboards improve insights",
    "A/B testing measures campaign success",
    "Customer segmentation improves marketing",
    "Fraud detection protects financial systems",
    "Personalization improves customer experience",
    "Deep learning requires large datasets",
    "Model evaluation ensures reliability",
    "Explainability builds model trust",
    "Ethics in AI ensures responsible innovation",
    "Data governance ensures data quality",
    "API integration connects multiple systems",
    "Microservices improve scalability",
    "Docker containers simplify deployment",
    "CI/CD pipelines automate software releases",
    "Serverless computing optimizes cost",
    "Edge AI brings inference closer to devices",
    "Federated learning protects privacy",
    "Prompt engineering improves LLM responses",
    "Tokenization breaks text into subwords",
    "Embeddings capture semantic meaning",
    "Vector search retrieves similar content",
    "RAG combines retrieval with generation",
    "LangChain simplifies LLM orchestration",
    "Agents coordinate multiple AI tasks",
    "Pinecone indexes large vector datasets",
    "Chroma provides lightweight vector storage",
    "Faiss accelerates approximate nearest neighbor search",
    "Sentence transformers create text embeddings",
    "Fine-tuning customizes pretrained models",
    "Quantization reduces model size",
    "Distillation transfers knowledge to smaller models",
    "LoRA fine-tunes large models efficiently",
    "RLHF aligns models with human feedback",
    "GPT models use autoregressive generation",
    "BERT uses bidirectional attention",
    "LLMs understand context and nuance",
    "OpenAI advances general-purpose AI",
    "Google trains foundation models at scale",
    "Meta focuses on open research models",
    "Anthropic emphasizes safety and alignment",
    "NVIDIA optimizes GPUs for deep learning",
    "Transformer architecture scales efficiently",
    "Self-attention connects distant tokens",
    "Batch normalization stabilizes learning",
    "Dropout prevents overfitting",
    "ReLU activation speeds convergence",
    "Adam optimizer adapts learning rates",
    "Learning rate scheduling improves training",
    "Data augmentation improves robustness",
    "Model checkpoints save progress",
    "Early stopping prevents overtraining",
    "Evaluation metrics guide improvement",
    "Precision and recall measure performance",
    "ROC curve visualizes trade-offs",
    "Confusion matrix summarizes predictions",
    "F1 score balances precision and recall",
    "AUC measures classification quality",
    "Regression minimizes squared errors",
    "Classification predicts categorical outcomes",
    "Clustering groups unlabeled data",
    "Anomaly detection identifies unusual patterns",
    "Dimensionality reduction simplifies data visualization",
    "Principal component analysis reduces redundancy",
    "t-SNE projects data into 2D space",
    "UMAP preserves local relationships",
    "Word2Vec learns from co-occurrence statistics",
    "Doc2Vec represents full sentences as vectors",
    "TF-IDF weighs word importance in documents",
    "BM25 scores document relevance in search engines",
]

c:\Users\kshit\OneDrive\Desktop\CodingNinjasAICourse\Rag\Vector Databases\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# STEP 1 — initialize vectordb (simple in-memory "db" + config)

# documents store (id -> raw text)
doc_db = {}

# embeddings store (id -> vector) [optional for your own bookkeeping]
vec_db = {}

# model & similarity choice
model_name = "sentence-transformers/paraphrase-MiniLM-L6-v2"  # 384-dim
model = SentenceTransformer(model_name)
dim = model.get_sentence_embedding_dimension()  # no need to embed yet

# IVF params
nlist = 16                     # number of clusters (cells)
metric = faiss.METRIC_INNER_PRODUCT  # we'll use cosine via L2-normalize + IP

In [3]:
# STEP 2 — Create index (if it doesn't exist)

# coarse quantizer for IVF
quantizer = faiss.IndexFlatIP(dim)   # IP pairs with our normalization

# IVF that stores raw vectors in each list ("Flat")
ivf = faiss.IndexIVFFlat(quantizer, dim, nlist, metric)

# Wrap with ID map so we can upsert using our own integer IDs
index = faiss.IndexIDMap2(ivf)

# NOTE: IVF needs training before first add(); we’ll do that in Step 6.
ivf_base = ivf  # Keep a reference to the IVF index

In [4]:
# STEP 3 — Access index (get its object)

def get_index():
    # In a real app, you might fetch from a registry / disk.
    return index  # we just return the in-memory object here

faiss_index = get_index()
faiss_index


<faiss.swigfaiss_avx2.IndexIDMap2; proxy of <Swig Object of type 'faiss::IndexIDMap2Template< faiss::Index > *' at 0x000001C717821CE0> >

In [5]:
ivf_base

<faiss.swigfaiss_avx2.IndexIVFFlat; proxy of <Swig Object of type 'faiss::IndexIVFFlat *' at 0x000001C732983810> >

In [6]:
# STEP 4 — Organize documents in [{id, doc}] format

# If you already have `texts`, map them to integer IDs (0..N-1)
doc_records = [{"id": i, "doc": txt} for i, txt in enumerate(texts)]

# Also fill our doc_db for convenience
for r in doc_records:
    doc_db[r["id"]] = r["doc"]
len(doc_records), doc_records[0]

(104, {'id': 0, 'doc': 'AI models are getting larger every year'})

In [7]:
# STEP 5 — Create embeddings and organize as [{id, [embedding]}]

# Encode all texts (cosine-ready: L2-normalize so IP == cosine)
X = model.encode([r["doc"] for r in doc_records], convert_to_numpy=True).astype(np.float32)
faiss.normalize_L2(X)  # in-place

ids = np.array([r["id"] for r in doc_records], dtype=np.int64)

# Optional: keep a Python-side vec_db mirror
for pid, vec in zip(ids, X):
    vec_db[pid] = vec

vec_records = [{"id": int(pid), "vector": vec.tolist()} for pid, vec in zip(ids, X)]
len(vec_records), vec_records[0]["id"]

(104, 0)

In [8]:
vec_records[0]

{'id': 0,
 'vector': [0.02850075624883175,
  -0.0493258573114872,
  -0.00477656489238143,
  -0.03708381950855255,
  0.014070727862417698,
  0.014333347789943218,
  -0.030122030526399612,
  0.016859352588653564,
  0.04295366257429123,
  0.043506499379873276,
  -0.008042380213737488,
  0.05526074767112732,
  0.005531408824026585,
  -0.0356060266494751,
  -0.10512852668762207,
  0.023230088874697685,
  -0.05596499517560005,
  -0.09660478681325912,
  -0.09688198566436768,
  0.044338617473840714,
  -0.014671248383820057,
  -0.12171399593353271,
  0.033407602459192276,
  -0.027206895872950554,
  0.053945813328027725,
  0.03134411573410034,
  -0.042549457401037216,
  -0.0021710325963795185,
  0.020848730579018593,
  -0.06934753805398941,
  0.0036189344245940447,
  0.01126501802355051,
  0.12776149809360504,
  0.02932952530682087,
  -0.014120941050350666,
  -0.06098482385277748,
  -0.0428207628428936,
  -0.05229830741882324,
  -0.049986060708761215,
  0.02069258689880371,
  -0.0255976431071758

In [11]:
# STEP 6 — Upsert (train IVF if needed, then add vectors with IDs)

base = faiss_index.index  # the underlying IVF index inside IDMap2

if not base.is_trained:
    base.train(X)  # train k-means centroids on your embeddings

# Add (or upsert). For true upsert semantics, remove existing IDs first.
# IndexIDMap2 supports add_with_ids; to replace, remove_ids then add_with_ids.
# For a first-time load, just add:
faiss_index.add_with_ids(X, ids)

faiss_index.ntotal

208

In [12]:
# Then modify search:
def search(query, topk=5, nprobe=4):
    # Use the stored reference
    ivf_base.nprobe = nprobe
    
    q = model.encode([query], convert_to_numpy=True).astype(np.float32)
    faiss.normalize_L2(q)
    D, I = faiss_index.search(q, topk)  # similarities (IP) and ids
    out = []
    for score, pid in zip(D[0], I[0]):
        if pid == -1:  # no result in that slot
            continue
        out.append((float(score), int(pid), doc_db[int(pid)]))
    return out

In [13]:
# try a couple of queries
for q in ["semantic vector search over documents",
          "Apple product announcements",
          "best time to contact customers"]:
    print("\nQuery:", q)
    for s, pid, txt in search(q, topk=5, nprobe=4):
        print(f"  score={s:.3f}  id={pid}  -> {txt}")


Query: semantic vector search over documents
  score=0.814  id=5  -> Vector databases improve semantic search
  score=0.814  id=5  -> Vector databases improve semantic search
  score=0.730  id=55  -> Vector search retrieves similar content
  score=0.730  id=55  -> Vector search retrieves similar content
  score=0.491  id=54  -> Embeddings capture semantic meaning

Query: Apple product announcements
  score=0.499  id=9  -> Apple launches new devices every September
  score=0.499  id=9  -> Apple launches new devices every September
  score=0.253  id=34  -> Business intelligence helps decision making
  score=0.253  id=34  -> Business intelligence helps decision making
  score=0.253  id=86  -> Evaluation metrics guide improvement

Query: best time to contact customers
  score=0.800  id=4  -> The best time to email customers is afternoon
  score=0.800  id=4  -> The best time to email customers is afternoon
  score=0.527  id=39  -> Personalization improves customer experience
  score=0.527 